# Day 5: Research Design, Validation, and Measurement Error

This lab is a bridge from methods to research design. The output is not a high-performing model; the output is a defensible plan for using text as evidence.

By the end you should be able to:

1. State what role text plays in a design.
2. Link a model output to a theoretical construct.
3. Sketch a simple causal diagram.
4. Simulate how measurement error can affect estimates.
5. Write a validation and reproducibility plan.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

pd.set_option('display.max_colwidth', 140)

## 1. Project design template

Fill this table for your own project. Keep answers short enough that another person can critique them.

In [ ]:
design = pd.DataFrame({
    'component': [
        'Research question',
        'Construct',
        'Unit of analysis',
        'Role of text',
        'Text model',
        'Model output',
        'Validation evidence',
        'Inferential claim',
        'Main risk'
    ],
    'your_answer': [
        '',
        '',
        '',
        'treatment / outcome / confounder / mediator / object',
        'dictionary / classifier / topic model / embedding / LLM',
        '',
        '',
        '',
        ''
    ]
})

design

## 2. Simple DAG sketch

Use this as a thinking tool. It is intentionally minimal: nodes and arrows. The point is to clarify assumptions, not to make a polished figure.

In [ ]:
def draw_dag(edges, positions=None, title='DAG sketch'):
    nodes = sorted(set([node for edge in edges for node in edge]))
    if positions is None:
        angles = np.linspace(0, 2 * np.pi, len(nodes), endpoint=False)
        positions = {node: (np.cos(a), np.sin(a)) for node, a in zip(nodes, angles)}

    plt.figure(figsize=(6, 4))
    ax = plt.gca()
    for node, (x, y) in positions.items():
        ax.scatter([x], [y], s=1800, color='white', edgecolor='black', zorder=2)
        ax.text(x, y, node, ha='center', va='center', zorder=3)
    for source, target in edges:
        x1, y1 = positions[source]
        x2, y2 = positions[target]
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1), arrowprops={'arrowstyle': '->', 'lw': 1.5})
    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()

edges = [('Confounder', 'Text measure'), ('Confounder', 'Outcome'), ('Text measure', 'Outcome')]
positions = {'Confounder': (0, 1), 'Text measure': (-1, 0), 'Outcome': (1, 0)}
draw_dag(edges, positions, title='Text as explanatory variable with confounding')

## 3. Measurement error simulation

Suppose the true text construct is binary, but our model measures it with error. How does that affect an estimated relationship with an outcome?

### Methodology formulas: validation and measurement error

The validation problem is that the observed text measure $M_i$ is an imperfect proxy for the latent construct $C_i$:

$$M_i = C_i + u_i.$$

For a binary classifier, two basic validation quantities are sensitivity and specificity:

$$\mathrm{sensitivity}=P(\hat{C}=1\mid C=1), \qquad \mathrm{specificity}=P(\hat{C}=0\mid C=0).$$

If the text measure is used as a treatment or explanatory variable, a simple estimating equation is

$$Y_i = \alpha + \tau M_i + \epsilon_i.$$

The estimand of interest is often defined in terms of the latent construct, not the noisy measurement:

$$Y_i = \alpha + \tau C_i + \epsilon_i.$$

The simulation tracks bias as

$$\mathrm{Bias}(\hat{\tau}) = E[\hat{\tau}] - \tau.$$

A strong project design therefore separates the construct, the observable measurement procedure, the validation data, and the final causal or descriptive estimand.


In [ ]:
rng = np.random.default_rng(42)
n = 2000

confounder = rng.normal(size=n)
true_text_construct = rng.binomial(1, 1 / (1 + np.exp(-0.8 * confounder)))
outcome = 1.5 * true_text_construct + 0.8 * confounder + rng.normal(size=n)

sim = pd.DataFrame({
    'confounder': confounder,
    'true_text_construct': true_text_construct,
    'outcome': outcome
})

sim.head()

In [ ]:
def misclassify_binary(z, sensitivity=0.85, specificity=0.85, rng=None):
    if rng is None:
        rng = np.random.default_rng(1)
    z = np.asarray(z)
    measured = z.copy()

    true_positive = z == 1
    true_negative = z == 0

    measured[true_positive] = rng.binomial(1, sensitivity, true_positive.sum())
    measured[true_negative] = rng.binomial(1, 1 - specificity, true_negative.sum())
    return measured

def estimate_effect(text_measure):
    X = pd.DataFrame({'text_measure': text_measure, 'confounder': sim['confounder']})
    model = LinearRegression().fit(X, sim['outcome'])
    return model.coef_[0]

true_effect_estimate = estimate_effect(sim['true_text_construct'])
true_effect_estimate

In [ ]:
rows = []
for sensitivity in [0.60, 0.70, 0.80, 0.90, 0.98]:
    for specificity in [0.60, 0.70, 0.80, 0.90, 0.98]:
        measured = misclassify_binary(
            sim['true_text_construct'],
            sensitivity=sensitivity,
            specificity=specificity,
            rng=np.random.default_rng(10)
        )
        rows.append({
            'sensitivity': sensitivity,
            'specificity': specificity,
            'estimated_effect': estimate_effect(measured)
        })

error_results = pd.DataFrame(rows)
error_results.head()

In [ ]:
pivot = error_results.pivot(index='sensitivity', columns='specificity', values='estimated_effect')
pivot

## 3a. Measurement-error heatmap

The same simulation is easier to interpret as a sensitivity surface. This is the kind of plot that can support a robustness discussion.

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='mako', cbar_kws={'label': 'Estimated effect'})
plt.title('Estimated effect under measurement error')
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.tight_layout()

## 3b. Bias relative to the true construct

Here we plot the bias induced by using the measured text variable instead of the true construct.

In [ ]:
error_results['bias'] = error_results['estimated_effect'] - true_effect_estimate
bias_pivot = error_results.pivot(index='sensitivity', columns='specificity', values='bias')

plt.figure(figsize=(7, 5))
sns.heatmap(bias_pivot, annot=True, fmt='.2f', cmap='vlag', center=0, cbar_kws={'label': 'Bias'})
plt.title('Bias from text-measurement error')
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.tight_layout()

In [ ]:
plt.figure(figsize=(6, 4))
for sensitivity, group in error_results.groupby('sensitivity'):
    plt.plot(group['specificity'], group['estimated_effect'], marker='o', label=f'sens={sensitivity}')
plt.axhline(true_effect_estimate, color='black', linestyle='--', label='using true construct')
plt.xlabel('Specificity')
plt.ylabel('Estimated effect of text measure')
plt.title('Measurement error changes estimates')
plt.legend()
plt.tight_layout()

### Additional demo: uncertainty from finite validation data

Measurement error is not only bias. With a finite sample, the estimated effect also varies across possible samples. This bootstrap makes sampling uncertainty visible.

In [ ]:
bootstrap_rng = np.random.default_rng(123)
measured_80_80 = misclassify_binary(
    sim['true_text_construct'],
    sensitivity=0.80,
    specificity=0.80,
    rng=np.random.default_rng(20)
)

bootstrap_rows = []
for draw in range(200):
    idx = bootstrap_rng.integers(0, len(sim), len(sim))
    boot = sim.iloc[idx].copy()
    boot_measure = np.asarray(measured_80_80)[idx]
    X = pd.DataFrame({'text_measure': boot_measure, 'confounder': boot['confounder']})
    model = LinearRegression().fit(X, boot['outcome'])
    bootstrap_rows.append({'draw': draw, 'estimated_effect': model.coef_[0]})

bootstrap_effects = pd.DataFrame(bootstrap_rows)
ci = bootstrap_effects['estimated_effect'].quantile([0.025, 0.975])

plt.figure(figsize=(7, 4))
sns.histplot(data=bootstrap_effects, x='estimated_effect', bins=25, color='#4C72B0')
plt.axvline(true_effect_estimate, color='black', linestyle='--', label='Estimate using true construct')
plt.axvline(ci.loc[0.025], color='#C44E52', linestyle=':', label='95% bootstrap interval')
plt.axvline(ci.loc[0.975], color='#C44E52', linestyle=':')
plt.title('Bootstrap variation with 80% sensitivity and specificity')
plt.xlabel('Estimated text-measure effect')
plt.legend()
plt.tight_layout()

pd.Series({'ci_low': ci.loc[0.025], 'ci_high': ci.loc[0.975], 'mean': bootstrap_effects['estimated_effect'].mean()})

## 4. Validation plan

A validation plan should say what evidence would make the measurement credible for the claim you want to make.

In [ ]:
validation_plan = pd.DataFrame({
    'question': [
        'What is the construct?',
        'What is the unit of analysis?',
        'What is the gold standard or comparison?',
        'Which errors matter most?',
        'What domain shifts are plausible?',
        'What robustness checks are needed?',
        'How will uncertainty be reported?'
    ],
    'plan': [''] * 7
})

validation_plan

## 5. Method choice matrix

Use this as a starting point, not as an automatic decision rule.

In [ ]:
method_matrix = pd.DataFrame([
    {
        'method': 'Dictionary',
        'use_when': 'Concept has explicit terms and transparency matters.',
        'main_risk': 'Misses context and synonyms.'
    },
    {
        'method': 'Supervised classifier',
        'use_when': 'You have labeled examples and care about prediction for a defined label.',
        'main_risk': 'Learns shortcuts from labels or sampling.'
    },
    {
        'method': 'Topic model',
        'use_when': 'You need exploratory structure in unlabeled documents.',
        'main_risk': 'Topics may be artifacts or unstable.'
    },
    {
        'method': 'Embedding measure',
        'use_when': 'You need semantic similarity, axes, or dense representations.',
        'main_risk': 'Axis or similarity may not match the construct.'
    },
    {
        'method': 'LLM annotation',
        'use_when': 'The concept needs contextual judgment and prompt/codebook design is clear.',
        'main_risk': 'Systematic errors, instability, and weak reproducibility.'
    },
    {
        'method': 'Fine-tuning',
        'use_when': 'You have enough high-quality labels and a clear reason baseline models fail.',
        'main_risk': 'Overfitting, compute cost, and weak out-of-domain validity.'
    }
])

method_matrix

## 5a. Method-choice view

This plot is intentionally simple: it turns the method matrix into a discussion aid for project clinics.

In [ ]:
method_scores = pd.DataFrame({
    'method': ['Dictionary', 'Supervised classifier', 'Topic model', 'Embedding measure', 'LLM annotation', 'Fine-tuning'],
    'label_needs': [1, 4, 1, 2, 2, 5],
    'interpretability': [5, 3, 3, 2, 2, 2],
    'infrastructure_cost': [1, 2, 2, 3, 4, 5]
}).set_index('method')

plt.figure(figsize=(8, 4))
sns.heatmap(method_scores, annot=True, cmap='YlGnBu', vmin=1, vmax=5, cbar_kws={'label': 'Low to high'})
plt.title('Method-choice tradeoffs for discussion')
plt.tight_layout()

method_scores

## 6. Final task

Prepare a five-minute project clinic note:

1. What claim do you want to make?
2. What text-derived measure supports the claim?
3. What validation evidence will you show?
4. What is the weakest part of the design?
5. What would a skeptical reviewer ask first?

In [ ]:
# Optional: after filling the tables, save them for your project notes.
# design.to_csv('my_project_design.csv', index=False)
# validation_plan.to_csv('my_validation_plan.csv', index=False)